In [1]:
!pip install kaggle

In [2]:
!mkdir ~/.kaggle

In [3]:
!cp kaggle.json ~/.kaggle/

In [4]:
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets download itamargr/dfdc-faces-of-the-train-sample

100% 3.63G/3.64G [00:45<00:00, 104MB/s] 
100% 3.64G/3.64G [00:45<00:00, 86.7MB/s]


In [ ]:
!unzip dfdc-faces-of-the-train-sample.zip

In [7]:
!kaggle datasets download kuvalgarg/dfdc-ensemble-model-weights

 97% 608M/624M [00:03<00:00, 195MB/s]
100% 624M/624M [00:03<00:00, 194MB/s]


In [8]:
!unzip dfdc-ensemble-model-weights.zip

Archive:  dfdc-ensemble-model-weights.zip
  inflating: efficientnet-b0.pth     
  inflating: efficientnet-b4.pth     
  inflating: efficientnet-b6.pth     
  inflating: efficientnet-b7.pth     
  inflating: mobilenet-v2.pth        
  inflating: ppg.pth                 
  inflating: resnet-50.pth           
  inflating: xception-net.pth        


In [9]:
!pip install efficientnet_pytorch

  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16428 sha256=fd54a97041ffe398472db582e408c2341a2912c7f6a185c78b341f41042519fd
  Stored in directory: /root/.cache/pip/wheels/03/3f/e9/911b1bc46869644912bda90a56bcf7b960f20b5187feea3baf
Successfully built efficientnet_pytorch


In [10]:
!pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 35.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 30.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.1 MB/s eta 0:00:00


In [11]:
! pip install torchviz

  Preparing metadata (setup.py) ... done
  Created wheel for torchviz: filename=torchviz-0.0.2-py3-none-any.whl size=4130 sha256=bc1ad73c9a8fe47d724c91c72d24075377b30cf68da8da6db8c754351ea75346
  Stored in directory: /root/.cache/pip/wheels/4c/97/88/a02973217949e0db0c9f4346d154085f4725f99c4f15a87094
Successfully built torchviz


In [74]:
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import os
from sklearn.metrics import accuracy_score
from PIL import Image
import io

In [75]:
from efficientnet_pytorch import EfficientNet
import timm

In [76]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Loading trained models with their weights
efficientnetb0_weights = torch.load('efficientnet-b0.pth', map_location=torch.device(device))
xceptionnet_weights = torch.load('xception-net.pth', map_location=torch.device(device))
efficientnetb7_weights = torch.load('efficientnet-b7.pth', map_location=torch.device(device))
ppg_weights = torch.load('ppg.pth', map_location=torch.device(device))

# Model Class Definitions

In [77]:
# Define EfficientNetB0 model architecture
class EfficientNetB0(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNetB0, self).__init__()
        self.efficientnet = EfficientNet.from_pretrained('efficientnet-b0', num_classes=num_classes)

    def forward(self, x):
        return self.efficientnet(x)

# Create an instance of the model
efficientnetb0_model = EfficientNetB0(num_classes=2)

# Load the saved weights into the model
efficientnetb0_model.load_state_dict(efficientnetb0_weights, strict=False)
efficientnetb0_model.to(device)
efficientnetb0_model.eval()
print('')

Loaded pretrained weights for efficientnet-b0



In [78]:
from torchviz import make_dot

# Create an instance of your model with the desired input size and number of classes
model = EfficientNetB0(num_classes=2)

# Assuming you have a sample input tensor
sample_input = torch.randn((1, 3, 224, 224))  # Adjust the input size (channels, height, width)

# Pass the sample input through the model to generate the computation graph
output = model(sample_input)

# Visualize the computation graph
graph = make_dot(output, params=dict(model.named_parameters()))

# Display the graph in a Jupyter Notebook or IPython
graph.view()

# Save the graph to a file (e.g., as a PDF or PNG)
graph.render("EfficientNetB0_graph")

Loaded pretrained weights for efficientnet-b0


'EfficientNetB0_graph.pdf'

In [79]:
# Define XceptionNet model architecture
class XceptionNet(nn.Module):
    def __init__(self, num_classes):
        super(XceptionNet, self).__init__()
        self.xceptionnet = timm.create_model('xception', pretrained=True)

        # Modify the output layer for binary classification
        num_ftrs = self.xceptionnet.fc.in_features
        self.xceptionnet.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.xceptionnet(x)

# Create an instance of the model
xceptionnet_model = XceptionNet(num_classes=2)

# Load the saved weights into the model
xceptionnet_model.load_state_dict(xceptionnet_weights, strict=False)
xceptionnet_model.to(device)
xceptionnet_model.eval()
print('')

/usr/local/lib/python3.10/dist-packages/timm/models/_factory.py:114: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


In [80]:
# Define EfficientNetB7 model architecture
class EfficientNetB7(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNetB7, self).__init__()
        self.efficientnet = EfficientNet.from_pretrained('efficientnet-b7', num_classes=num_classes)

    def forward(self, x):
        return self.efficientnet(x)

# Create an instance of the model
efficientnetb7_model = EfficientNetB7(num_classes=2)

# Load the saved weights into the model
efficientnetb7_model.load_state_dict(efficientnetb7_weights, strict=False)
efficientnetb7_model.to(device)
efficientnetb7_model.eval()
print('')

Loaded pretrained weights for efficientnet-b7



In [81]:
# Define PPG model architecture
class Attention_mask(nn.Module):
    def __init__(self):
        super(Attention_mask, self).__init__()

    def forward(self, x):
        xsum = torch.sum(x, dim=2, keepdim=True)
        xsum = torch.sum(xsum, dim=3, keepdim=True)
        xshape = tuple(x.size())
        return x / xsum * xshape[2] * xshape[3] * 0.5

    def get_config(self):
        """May be generated manually. """
        config = super(Attention_mask, self).get_config()
        return config

class DeepPhys(nn.Module):

    def __init__(self, in_channels=3, nb_filters1=32, nb_filters2=64, kernel_size=3, dropout_rate1=0.25,
                 dropout_rate2=0.5, pool_size=(2, 2), nb_dense=128, img_size=36):
        """Definition of DeepPhys.
        Args:
        in_channels: the number of input channel. Default: 3
        img_size: height/width of each frame. Default: 36.
        Returns:
        DeepPhys model.
        """
        super(DeepPhys, self).__init__()

        # self.final_dense_1 = nn.Linear(16384, 128, bias=True)

        self.in_channels = in_channels
        self.kernel_size = kernel_size
        self.dropout_rate1 = dropout_rate1
        self.dropout_rate2 = dropout_rate2
        self.pool_size = pool_size
        self.nb_filters1 = nb_filters1
        self.nb_filters2 = nb_filters2
        self.nb_dense = nb_dense

        # Motion branch convs
        self.motion_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size, padding=(1, 1),
                                    bias=True)
        self.motion_conv2 = nn.Conv2d(self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, bias=True)
        self.motion_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size, padding=(1, 1),
                                    bias=True)
        self.motion_conv4 = nn.Conv2d(self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, bias=True)

        # Apperance branch convs
        self.apperance_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size,
                                        padding=(1, 1), bias=True)
        self.apperance_conv2 = nn.Conv2d(self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, bias=True)
        self.apperance_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size,
                                        padding=(1, 1), bias=True)
        self.apperance_conv4 = nn.Conv2d(self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, bias=True)

        # Attention layers
        self.apperance_att_conv1 = nn.Conv2d(self.nb_filters1, 1, kernel_size=1, padding=(0, 0), bias=True)
        self.attn_mask_1 = Attention_mask()
        self.apperance_att_conv2 = nn.Conv2d(self.nb_filters2, 1, kernel_size=1, padding=(0, 0), bias=True)
        self.attn_mask_2 = Attention_mask()

        # Avg pooling
        self.avg_pooling_1 = nn.AvgPool2d(self.pool_size)
        self.avg_pooling_2 = nn.AvgPool2d(self.pool_size)
        self.avg_pooling_3 = nn.AvgPool2d(self.pool_size)

        # Dropout layers
        self.dropout_1 = nn.Dropout(self.dropout_rate1)
        self.dropout_2 = nn.Dropout(self.dropout_rate1)
        self.dropout_3 = nn.Dropout(self.dropout_rate1)
        self.dropout_4 = nn.Dropout(self.dropout_rate2)

        # Dense layers
        if img_size == 36:
            self.final_dense_1 = nn.Linear(3136, self.nb_dense, bias=True)
        elif img_size == 72:
            self.final_dense_1 = nn.Linear(16384, self.nb_dense, bias=True)
        elif img_size == 96:
            self.final_dense_1 = nn.Linear(30976, self.nb_dense, bias=True)
        else:
            raise Exception('Unsupported image size')

        # Final dense layer with a single neuron
        self.final_dense_2 = nn.Linear(self.nb_dense, 1, bias=True)
    def forward(self, inputs, params=None):
        diff_input = inputs[:, :3, :, :]
        raw_input = diff_input

        d1 = torch.tanh(self.motion_conv1(diff_input))
        d2 = torch.tanh(self.motion_conv2(d1))

        r1 = torch.tanh(self.apperance_conv1(raw_input))
        r2 = torch.tanh(self.apperance_conv2(r1))

        g1 = torch.sigmoid(self.apperance_att_conv1(r2))
        g1 = self.attn_mask_1(g1)
        gated1 = d2 * g1

        d3 = self.avg_pooling_1(gated1)
        d4 = self.dropout_1(d3)

        r3 = self.avg_pooling_2(r2)
        r4 = self.dropout_2(r3)

        d5 = torch.tanh(self.motion_conv3(d4))
        d6 = torch.tanh(self.motion_conv4(d5))

        r5 = torch.tanh(self.apperance_conv3(r4))
        r6 = torch.tanh(self.apperance_conv4(r5))

        g2 = torch.sigmoid(self.apperance_att_conv2(r6))
        g2 = self.attn_mask_2(g2)
        gated2 = d6 * g2

        d7 = self.avg_pooling_3(gated2)
        d8 = self.dropout_3(d7)
        d9 = d8.view(d8.size(0), -1)
        d10 = torch.tanh(self.final_dense_1(d9))
        d11 = self.dropout_4(d10)
        out = torch.sigmoid(self.final_dense_2(d11))  # Clone and detach to keep in the computation graph
        return out

# In case the keys have the 'module.' prefix, remove it to match the keys in the current model
if list(ppg_weights.keys())[0].startswith('module.'):
    ppg_weights = {k[7:]: v for k, v in ppg_weights.items()}

# Initialize the model with the same parameters as the original model
ppg_model = DeepPhys(in_channels=3, nb_filters1=32, nb_filters2=64, kernel_size=3,
                         dropout_rate1=0.8, dropout_rate2=0.8, pool_size=(2, 2), nb_dense=128, img_size=72)

# Load the matched state_dict into the model
ppg_model.load_state_dict(ppg_weights)

ppg_model.to(device)
ppg_model.eval()
print('')

In [82]:
# # Define data transformations for inference
# data_transforms = transforms.Compose([
#     transforms.Resize(224),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
# ])

In [83]:
# For PPG Specifically

# Define data transformations for inference
data_transforms = transforms.Compose([
    transforms.Resize(72),
    transforms.CenterCrop(72),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Testing

In [84]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import random

# Path to the test dataset directory
test_data_dir = 'validation'

# Create a test dataset using ImageFolder
test_dataset = ImageFolder(test_data_dir, transform=data_transforms)

# Determine the size of the subset (30%)
subset_size = int(0.3 * len(test_dataset))

# Generate random indices for the subset
random_indices = random.sample(range(len(test_dataset)), subset_size)

# Create a subset of the test dataset
test_subset = Subset(test_dataset, random_indices)

# Create a dataloader for the test dataset
batch_size = 32  # Adjust this based on your available memory
test_dataloader = DataLoader(test_subset, batch_size=batch_size, shuffle=True)

# Set your models to evaluation mode
efficientnetb0_model.eval()
xceptionnet_model.eval()
efficientnetb7_model.eval()
ppg_model.eval()

# Move the models to the same device as your data
efficientnetb0_model.to(device)
xceptionnet_model.to(device)
efficientnetb7_model.to(device)
ppg_model.to(device)

print('')

In [85]:
# correct_predictions = 0
# total_samples = 0

# # Iterate through the test dataloader
# with torch.no_grad():
#     for inputs, labels in test_dataloader:
#         inputs = inputs.to(device)
#         labels = labels.to(device)

#         outputs = efficientnetb0_model(inputs)
#         _, preds = torch.max(outputs, 1)

#         correct_predictions += torch.sum(preds == labels.data)
#         total_samples += labels.size(0)

# # Calculate accuracy
# test_accuracy = correct_predictions.double() / total_samples
# print(f'EfficientNetB0 \nCorrect Predictions: {correct_predictions} \nTotal Predictions: {total_samples} \nTest Accuracy: {test_accuracy:.4f}')

In [86]:
# correct_predictions = 0
# total_samples = 0

# # Iterate through the test dataloader
# with torch.no_grad():
#     for inputs, labels in test_dataloader:
#         inputs = inputs.to(device)
#         labels = labels.to(device)

#         outputs = xceptionnet_model(inputs)
#         _, preds = torch.max(outputs, 1)

#         correct_predictions += torch.sum(preds == labels.data)
#         total_samples += labels.size(0)

# # Calculate accuracy
# test_accuracy = correct_predictions.double() / total_samples
# print(f'XceptionNet \nCorrect Predictions: {correct_predictions} \nTotal Predictions: {total_samples} \nTest Accuracy: {test_accuracy:.4f}')

In [87]:
# correct_predictions = 0
# total_samples = 0

# # Iterate through the test dataloader
# with torch.no_grad():
#     for inputs, labels in test_dataloader:
#         inputs = inputs.to(device)
#         labels = labels.to(device)

#         outputs = efficientnetb7_model(inputs)
#         _, preds = torch.max(outputs, 1)

#         correct_predictions += torch.sum(preds == labels.data)
#         total_samples += labels.size(0)

# # Calculate accuracy
# test_accuracy = correct_predictions.double() / total_samples
# print(f'EfficientNetB7 \nCorrect Predictions: {correct_predictions} \nTotal Predictions: {total_samples} \nTest Accuracy: {test_accuracy:.4f}')

In [88]:
# # Define a threshold for binary classification
# threshold = 0.5

# correct_predictions = 0
# total_samples = 0

# # Iterate through the test dataloader
# with torch.no_grad():
#     for inputs, labels in test_dataloader:
#         inputs = inputs.to(device)
#         labels = labels.to(device)

#         outputs = ppg_model(inputs)
#         predicted_labels = (outputs >= threshold).int()  # Apply threshold for binary classification

#         correct_predictions += torch.sum(predicted_labels == labels.data)
#         total_samples += labels.size(0)

# # Calculate accuracy
# test_accuracy = correct_predictions.double() / total_samples
# print(f'PPG \nCorrect Predictions: {correct_predictions} \nTotal Predictions: {total_samples} \nTest Accuracy: {test_accuracy:.4f}')

# Ensemble [0.10, 0.20, 0.70] -> [0.10, 0.15, 0.25, 0.50]

In [92]:
class EnsembleModel(nn.Module):
    def __init__(self, model1, model2, model3, model4, weights):
        super(EnsembleModel, self).__init__()
        self.model1 = model1
        self.model2 = model2
        self.model3 = model3
        self.model4 = model4
        self.weights = weights

    def forward(self, x):
        preds1 = self.model1(x)
        preds2 = self.model2(x)
        preds3 = self.model3(x)
        preds4 = self.model4(x)

        # Apply softmax to convert logits to probabilities
        preds1_probs = torch.softmax(preds1, dim=1)
        preds2_probs = torch.softmax(preds2, dim=1)
        preds3_probs = torch.softmax(preds3, dim=1)
        preds4_probs = torch.softmax(preds4, dim=1)

        # Calculate the ensemble prediction (weighted average)
        weighted_predictions = (
            (preds1_probs * self.weights[0])
            + (preds2_probs * self.weights[1])
            + (preds3_probs * self.weights[2])
            + (preds4_probs * self.weights[3])
        )

        return weighted_predictions

# Create an instance of the ensemble model
ensemble_model = EnsembleModel(efficientnetb0_model, xceptionnet_model, ppg_model, efficientnetb7_model, [0.10, 0.20, 0.0, 0.70])

# Save the ensemble model to a file
torch.save(ensemble_model.state_dict(), 'ensemble_model.pth')

In [93]:
correct_predictions = 0
total_samples = 0

# Set the threshold for binary classification
threshold = 0.5

# Iterate through the test dataloader
with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Make predictions using the ensemble model
        ensemble_predictions = ensemble_model(inputs)

        # Apply the threshold for binary classification
        binary_predictions = (ensemble_predictions[:, 1] > threshold).int()

        correct_predictions += torch.sum(binary_predictions == labels.data)
        total_samples += labels.size(0)

# Calculate accuracy
test_accuracy = correct_predictions.double() / total_samples
print(f'Ensemble \nCorrect Predictions: {correct_predictions} \nTotal Predictions: {total_samples} \nTest Accuracy: {test_accuracy:.4f}')

Ensemble 
Correct Predictions: 1909 
Total Predictions: 9238 
Test Accuracy: 0.2066


# Sample Prediction

In [91]:
# # Load an image for testing (replace with your own image)
# image = Image.open('test_image.png')  # Provide the path to your image

# # Preprocess the image to make it compatible with the model
# image = data_transforms(image).unsqueeze(0)  # Add an extra dimension to represent batch size

# # Move the preprocessed image to the same device as your model
# image = image.to(device)  # Assuming 'device' is defined as 'cuda:0' or 'cpu'

# # Use the model to make predictions
# with torch.no_grad():
#     output = ensemble_model(image)

# # Assuming you have two classes, you can get the predicted class:
# predicted_class = output.argmax().item()

# # Interpret the prediction
# if predicted_class == 0:
#     prediction = "Real"
# else:
#     prediction = "DeepFake"

# print(f'Predicted Class: {predicted_class} ({prediction})')